#### Load Data

In [1]:
import json

selfcheck_scores_unigram = {}
selfcheck_scores_bigram = {}
selfcheck_scores_trigram = {}
selfcheck_scores_fourgram = {}
selfcheck_scores_fivegram = {}
selfcheck_scores_nli = {}
selfcheck_scores_bertscore = {}
selfcheck_scores_prompt_llama = {}
selfcheck_scores_prompt_solar_pro = {}

with open('data/selfcheck_scores_unigram.json', 'r') as file:
    selfcheck_scores_unigram = json.load(file)
with open('data/selfcheck_scores_bigram.json', 'r') as file:
    selfcheck_scores_bigram = json.load(file)
with open('data/selfcheck_scores_trigram.json', 'r') as file:
    selfcheck_scores_trigram = json.load(file)
with open('data/selfcheck_scores_fourgram.json', 'r') as file:
    selfcheck_scores_fourgram = json.load(file)
with open('data/selfcheck_scores_fivegram.json', 'r') as file:
    selfcheck_scores_fivegram = json.load(file)
with open('data/selfcheck_scores_nli.json', 'r') as file:
    selfcheck_scores_nli = json.load(file)
with open('data/selfcheck_scores_bertscore.json', 'r') as file:
    selfcheck_scores_bertscore = json.load(file)
with open('data/selfcheck_scores_prompt_Llama-3.2-1B.json', 'r') as file:
    selfcheck_scores_prompt_llama = json.load(file)
with open('data/selfcheck_scores_prompt_solar_pro.json', 'r') as file:
    selfcheck_scores_prompt_solar_pro = json.load(file)

In [2]:
import math

def logistic_transform(value):
    if isinstance(value, list):
        return [1 / (1 + math.exp(-x)) if x != float('inf') else 1.0 for x in value]
    else:
        return 1 / (1 + math.exp(-value)) if value != float('inf') else 1.0

def process_ngram_score(scores):
    scores_sent_avg_final = {}
    scores_sent_max_final = {}
    scores_doc_avg_final = {}
    scores_doc_avg_max_final = {}

    for idx in scores:
        scores_sent = scores[idx]['sent_level']
        scores_doc = scores[idx]['doc_level']

        scores_sent_avg = logistic_transform(scores_sent['avg_neg_logprob'])
        scores_sent_max = logistic_transform(scores_sent['max_neg_logprob'])
        score_doc_avg = logistic_transform(scores_doc['avg_neg_logprob'])
        score_doc_avg_max = logistic_transform(scores_doc['avg_max_neg_logprob'])

        scores_sent_avg_final[int(idx)] = scores_sent_avg
        scores_sent_max_final[int(idx)] = scores_sent_max
        scores_doc_avg_final[int(idx)] = score_doc_avg
        scores_doc_avg_max_final[int(idx)] = score_doc_avg_max
    return [scores_sent_avg_final, scores_sent_max_final, scores_doc_avg_final, scores_doc_avg_max_final]
    
scores_unigram = process_ngram_score(selfcheck_scores_unigram)
scores_bigram = process_ngram_score(selfcheck_scores_bigram)
scores_trigram = process_ngram_score(selfcheck_scores_trigram)
scores_fourgram = process_ngram_score(selfcheck_scores_fourgram)
scores_fivegram = process_ngram_score(selfcheck_scores_fivegram)

In [3]:
def process_score(scores):
    res = {}

    for idx in scores:
        res[int(idx)] = scores[idx]
    return res

scores_nli = process_score(selfcheck_scores_nli)
scores_bertscore = process_score(selfcheck_scores_bertscore)
scores_prompt_llama = process_score(selfcheck_scores_prompt_llama)
scores_prompt_solar_pro = process_score(selfcheck_scores_prompt_solar_pro)

In [4]:
import json
import numpy as np

with open("data/dataset_v3.json", "r") as f:
    content = f.read()
dataset = json.loads(content)
print("The lenght of the dataset: {}".format(len(dataset)))
print(dataset[0].keys())


label_mapping = {
    'accurate': 0.0,
    'minor_inaccurate': 0.5,
    'major_inaccurate': 1.0,
}

human_label_detect_False   = {}
human_label_detect_True    = {}
human_label_detect_False_h = {}

for i_ in range(len(dataset)):
    dataset_i = dataset[i_]
    idx = dataset_i["wiki_bio_test_idx"]
    
    raw_label = np.array([label_mapping[x] for x in dataset_i['annotation']])
    human_label_detect_False[idx] = (raw_label > 0.499).astype(np.int32).tolist()
    human_label_detect_True[idx] = (raw_label < 0.499).astype(np.int32).tolist()
    average_score = np.mean(raw_label)
    if (average_score < 0.99):
        human_label_detect_False_h[idx] = (raw_label > 0.99).astype(np.int32).tolist()
print("Length of False:", len(human_label_detect_False))
print("Length of True:", len(human_label_detect_True)) 
print("Length of False_h:", len(human_label_detect_False_h))

indices = [x['wiki_bio_test_idx'] for x in dataset] 

The lenght of the dataset: 238
dict_keys(['gpt3_text', 'wiki_bio_text', 'gpt3_sentences', 'annotation', 'wiki_bio_test_idx', 'gpt3_text_samples'])
Length of False: 238
Length of True: 238
Length of False_h: 206


## Experiments

In [5]:
arr_false = []
arr_false_h = []
arr_true = []

for v in human_label_detect_False.values():
    arr_false.extend(v)
for v in human_label_detect_False_h.values():
    arr_false_h.extend(v)
for v in human_label_detect_True.values():
    arr_true.extend(v)
    
random_baseline_false = np.mean(arr_false)
random_baseline_false_h = np.mean(arr_false_h)
random_baseline_true = np.mean(arr_true)

labels_passage_avg = []

for id in indices:
    labels_passage_avg.append(np.mean(human_label_detect_False[id]))

print("Random baseline false:", np.round(random_baseline_false, 2))
print("Random baseline false h:", np.round(random_baseline_false_h, 2))
print("Random baseline true:", np.round(random_baseline_true, 2))

Random baseline false: 0.73
Random baseline false h: 0.3
Random baseline true: 0.27


In [6]:
from sklearn.metrics import precision_recall_curve, auc

def unroll_pred(scores, indices):
    unrolled = []
    
    for idx in indices:
        unrolled.extend(scores[idx])
    return unrolled

def get_PR_with_human_labels(preds, human_labels, pos_label=1, oneminus_pred=False):
    indices = [k for k in human_labels.keys()]
    unroll_preds = unroll_pred(preds, indices)
    if oneminus_pred:
        unroll_preds = [1.0-x for x in unroll_preds]
    unroll_labels = unroll_pred(human_labels, indices)
    
    assert(len(unroll_preds) == len(unroll_labels))

    # print("Length: ", len(unroll_preds))
    p, r, threshold = precision_recall_curve(unroll_labels, unroll_preds, pos_label=pos_label)
    return p, r, threshold

def get_AUC(p, r):
    return (auc(r, p)*100)

In [7]:
import numpy as np
from scipy import stats

def sample(scores, labels, oneminus_pred=False):
    prec, rec, threshold = get_PR_with_human_labels(scores, labels, pos_label=1, oneminus_pred=oneminus_pred)
    return np.round(get_AUC(prec, rec), 2)

def getScorePassageNgram(scores):
    scores_passage = []

    for id in indices:
        # scores_passage.append(np.mean(scores[id]))    
        scores_passage.append(scores[id])

    score_pearsonnr = stats.pearsonr(scores_passage, labels_passage_avg)
    score_spearmanr = stats.spearmanr(scores_passage, labels_passage_avg)
    return np.round(score_pearsonnr[0] * 100, 2), np.round(score_spearmanr[0] * 100, 2)

def updateResultNgram(df, name, scores, scores_passage):
    df = df._append({"Method":  name, 
                  "NonFact": sample(scores, human_label_detect_False), 
                  "NonFact-H": sample(scores, human_label_detect_False_h), 
                  "Factual": sample(scores, human_label_detect_True, oneminus_pred=True),
                  "Pearson": scores_passage[0],
                  "Spearman": scores_passage[1]}, ignore_index = True)
    return df

def addNgramModuleToResult(df, name, scores):
    scores_passage_avg = getScorePassageNgram(scores[2])
    scores_passage_max = getScorePassageNgram(scores[3])

    df = updateResultNgram(df, name+"-Sent Avg", scores[0], scores_passage_avg)
    df = updateResultNgram(df, name+"-Sent Max", scores[1], scores_passage_max)
    return df

#### To Table

In [9]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.DataFrame(columns=["Method", "NonFact", "NonFact-H", "Factual", "Pearson", "Spearman"])
df = addNgramModuleToResult(df, "Unigram", scores_unigram)
df = addNgramModuleToResult(df, "Bigram", scores_bigram)
df = addNgramModuleToResult(df, "Trigram", scores_trigram)
df = addNgramModuleToResult(df, "Four-gram", scores_fourgram)
df = addNgramModuleToResult(df, "Five-gram", scores_fivegram)

res = pd.DataFrame(columns=["Method", "NonFact", "NonFact-H", "Factual", "Pearson", "Spearman"])

ngram_avg_methods = df[df["Method"].str.endswith('Avg')]
ngram_max_methods = df[df['Method'].str.endswith('Max')]

def getScorePassage(scores):
    scores_passage = []

    for id in indices:
        scores_passage.append(np.mean(scores[id]))    
        # scores_passage.append(scores[id])

    score_pearsonnr = stats.pearsonr(scores_passage, labels_passage_avg)
    score_spearmanr = stats.spearmanr(scores_passage, labels_passage_avg)
    return np.round(score_pearsonnr[0] * 100, 2), np.round(score_spearmanr[0] * 100, 2)

def updateResult(df, name, scores):
    scores_passage = getScorePassage(scores)

    df = df._append({"Method":  name, 
                  "NonFact": sample(scores, human_label_detect_False), 
                  "NonFact-H": sample(scores, human_label_detect_False_h), 
                  "Factual": sample(scores, human_label_detect_True, oneminus_pred=True),
                  "Pearson": scores_passage[0],
                  "Spearman": scores_passage[1]}, ignore_index = True)
    return df

res = res._append({
    "Method": "Random Baseline",
    "NonFact": np.round(random_baseline_false * 100, 2),
    "NonFact-H": np.round(random_baseline_false_h * 100, 2),
    "Factual": np.round(random_baseline_true * 100, 2)}, ignore_index = True)
res = pd.concat([res, ngram_avg_methods])
res = pd.concat([res, ngram_max_methods])
res = updateResult(res, "NLI", scores_nli)
res = updateResult(res, "Bertscore", scores_bertscore)
res = updateResult(res, "Llama-3.2-1B", scores_prompt_llama)
res = updateResult(res, "Solar Pro", scores_prompt_solar_pro)
res.to_csv("data/results.csv")

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_409132\362722127.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = df._append({"Method":  name,
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_409132\2992626301.py:39: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  res = res._append({
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_409132\2992626301.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude 

In [10]:
res

,Method,NonFact,NonFact-H,Factual,Pearson,Spearman
0,Random Baseline,72.96,29.72,27.04,NaN,NaN
1,Unigram-Sent Avg,81.52,40.33,41.78,35.26,36.72
2,Bigram-Sent Avg,82.83,44.11,52.67,50.44,50.86
3,Trigram-Sent Avg,83.27,43.79,53.85,52.94,53.82
4,Four-gram-Sent Avg,83.21,42.45,53.89,51.53,53.57
5,Five-gram-Sent Avg,82.85,41.04,53.63,50.95,53.53
6,Unigram-Sent Max,85.64,41.05,58.44,60.97,63.42
7,Bigram-Sent Max,85.04,38.97,58.11,57.11,62.40
8,Trigram-Sent Max,84.49,36.45,56.95,51.11,54.86
9,Four-gram-Sent Max,83.71,35.57,55.65,48.63,49.26
